In [1]:
import os
from autogen.agentchat import UserProxyAgent, AssistantAgent, GroupChat, GroupChatManager
from autogen.coding import LocalCommandLineCodeExecutor
from dotenv import load_dotenv
from openai import AzureOpenAI
import json
import pandas as pd
import numpy as np
from datetime import datetime
from collections import Counter
from pydantic import BaseModel
from typing import Literal
load_dotenv()

azure_gpt4o = {
    "api_type": "azure",
    "model": os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'),
    "api_key": os.getenv('OPENAI_API_KEY'),
    "base_url": os.getenv('AZURE_OPENAI_ENDPOINT'),
    "api_version": os.getenv('OPENAI_API_VERSION')
}

flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.


In [2]:
#print(os.getenv('OPENAI_API_VERSION'))
participant_id = 5
transcripts_dir = os.path.join(os.getcwd(), 'transcripts')
data_dir    = os.path.join(transcripts_dir, f"participant{participant_id}")
#blandai_data_dir = os.path.join(os.getcwd(), 'blandai-data')
codebook_file = os.path.join(transcripts_dir, 'codebook.json')
transcripts_file = os.path.join(data_dir, 'blandai_transcripts.json')
#sample_output_file = os.path.join(data_dir, 'example_output.csv')
#print(sample_output_file)
#print(codebook_file)
#print(blandai_data_dir)
#print(transcripts_file)

In [3]:
def get_QA_and_codebook(codebook_file):
    qa_list = []
    with open(codebook_file, 'r') as file:
        codebook = json.load(file)
        question_count = 1
        for code, content in codebook.items():  
            question = content['question']
            qa_list.append(f"Question {question_count}: {question}\nResponse Options: ")
            ans_list = []
            for ans, ans_id in content['clean_response_text_to_id'].items():
                ans_list.append(f"{ans}")
            #if code in ['HH01S', 'HH25S', 'HH612S', 'HH1317S', 'HH18OVS', 'PHYS11_TEMP']:
            #    ans_list = ['Numeric Value']
                
            qa_list.append('; '.join(ans_list))
            qa_list.append("\n")
            question_count += 1
    return ''.join(qa_list), codebook
QA_details, codebook = get_QA_and_codebook(codebook_file)
question_count = len(codebook)
print(QA_details)
print(question_count)

Question 1: What is your current age?
Response Options: 18-24; 25-34; 35-44; 45-54; 55-64; 65-74; 75+; Under 18
Question 2: Are you male or female?
Response Options: Male; Female; REFUSED
Question 3: What race or races you consider yourself to be? You can say multiple races.
Response Options: White; Black or African American; American Indian or Alaska Native; Asian Indian; Chinese; Filipino; Japanese; Korean; Vietnamese; Other Asian; Native Hawaiian; Guamanian or Chamorro; Samoan; Other Pacific Islander; Some other race; REFUSED
Question 4: What was your total HOUSEHOLD income in 2019?
Response Options: Under $10,000; $10,000 to under $20,000; $20,000 to under $30,000; $30,000 to under $40,000; $40,000 to under $50,000; $50,000 to under $75,000; $75,000 to under $100,000; $100,000 to under $150,000; $150,000 or more; DON'T KNOW; REFUSED
Question 5: What is the highest level of school you have completed?
Response Options: No formal education; 1st, 2nd, 3rd, or 4th grade; 5th or 6th grad

In [4]:
class SurveyResponses(BaseModel):  
    # Responses for questions 1-4  
    response_to_question_1: Literal[  
        "18-24",   
        "25-34",   
        "35-44",   
        "45-54",   
        "55-64",   
        "65-74",   
        "75+",   
        "Under 18",   
        "OTHER"  
    ]  
    response_to_question_2: Literal["Male", "Female"]  
    response_to_question_3: Literal[  
        "White",   
        "Black or African American",   
        "American Indian or Alaska Native",   
        "Asian Indian",   
        "Chinese",   
        "Filipino",   
        "Japanese",   
        "Korean",   
        "Vietnamese",   
        "Other Asian",   
        "Native Hawaiian",   
        "Guamanian or Chamorro",   
        "Samoan",   
        "Other Pacific Islander",   
        "Some other race",   
        "REFUSED",   
        "OTHER"  
    ]  
    response_to_question_4: Literal[  
        "Under $10,000",   
        "$10,000 to under $20,000",   
        "$20,000 to under $30,000",   
        "$30,000 to under $40,000",   
        "$40,000 to under $50,000",   
        "$50,000 to under $75,000",   
        "$75,000 to under $100,000",   
        "$100,000 to under $150,000",   
        "$150,000 or more",   
        "DON'T KNOW",   
        "REFUSED",   
        "OTHER"  
    ]  
  
    # Responses for questions 5-11  
    response_to_question_5: Literal[  
        "No formal education",   
        "1st, 2nd, 3rd, or 4th grade",   
        "5th or 6th grade",   
        "7th or 8th grade",   
        "9th grade",   
        "10th grade",   
        "11th grade",   
        "12th grade - NO DIPLOMA",   
        "High school graduate - high school diploma or the equivalent",   
        "Some college, no degree",   
        "Associate degree",   
        "Bachelor's degree",   
        "Master's degree",   
        "Professional or Doctorate degree",   
        "REFUSED",   
        "OTHER"  
    ]  
    response_to_question_6: Literal[  
        "One person, I live by myself",   
        "Two persons",   
        "Three persons",   
        "Four persons",   
        "Five persons",   
        "Six or more persons",   
        "REFUSED",   
        "OTHER"  
    ]  
    response_to_question_7: Literal["DON'T KNOW", "REFUSED", "OTHER"] | int  
    response_to_question_8: Literal["DON'T KNOW", "REFUSED", "OTHER"] | int  
    response_to_question_9: Literal["DON'T KNOW", "REFUSED", "OTHER"] | int  
    response_to_question_10: Literal["DON'T KNOW", "REFUSED", "OTHER"] | int  
    response_to_question_11: Literal["DON'T KNOW", "REFUSED", "OTHER"] | int  
  
    # Responses for questions 12-13  
    response_to_question_12: Literal[  
        "Basically every day",   
        "A few times a week",   
        "A few times a month",   
        "Once a month",   
        "Not at all",   
        "Not sure",   
        "REFUSED",   
        "OTHER"  
    ]  
    response_to_question_13: Literal[  
        "Basically every day",   
        "A few times a week",   
        "A few times a month",   
        "Once a month",   
        "Not at all",   
        "Not sure",   
        "REFUSED",   
        "OTHER"  
    ]  
  
    # Responses for questions 14-18  
    response_to_question_14: Literal[  
        "Not at all or less than 1 day",   
        "1-2 days",   
        "3-4 days",   
        "5-7 days",   
        "DON'T KNOW",   
        "REFUSED",   
        "OTHER"  
    ]  
    response_to_question_15: Literal[  
        "Not at all or less than 1 day",   
        "1-2 days",   
        "3-4 days",   
        "5-7 days",   
        "DON'T KNOW",   
        "REFUSED",   
        "OTHER"  
    ]  
    response_to_question_16: Literal[  
        "Not at all or less than 1 day",   
        "1-2 days",   
        "3-4 days",   
        "5-7 days",   
        "DON'T KNOW",   
        "REFUSED",   
        "OTHER"  
    ]  
    response_to_question_17: Literal[  
        "Not at all or less than 1 day",   
        "1-2 days",   
        "3-4 days",   
        "5-7 days",   
        "DON'T KNOW",   
        "REFUSED",   
        "OTHER"  
    ]  
    response_to_question_18: Literal[  
        "Not at all or less than 1 day",   
        "1-2 days",   
        "3-4 days",   
        "5-7 days",   
        "DON'T KNOW",   
        "REFUSED",   
        "OTHER"  
    ]  
  
    # Responses for questions 19-21  
    response_to_question_19: Literal[  
        "Yes, I worked for someone else for wages, salary, piece rate, commission, tips, or payments 'in kind,' for example, food or lodging received as payment for work performed",   
        "Yes, I worked as self-employed in my own business, professional practice, or farm",   
        "No, I did not work for pay last week",   
        "DON'T KNOW",   
        "REFUSED",   
        "OTHER"  
    ]  
    response_to_question_20: Literal[  
        "Excellent",   
        "Very good",   
        "Good",   
        "Fair",   
        "Poor",   
        "DON'T KNOW",   
        "REFUSED",   
        "OTHER"  
    ]  
    response_to_question_21: Literal[  
        "Yes",   
        "No",   
        "Not sure",   
        "REFUSED",   
        "OTHER"  
    ]  
  
    # Responses for questions 22-31  
    response_to_question_22: Literal["Yes", "No", "Not sure", "REFUSED", "OTHER"]  
    response_to_question_23: Literal["Yes", "No", "Not sure", "REFUSED", "OTHER"]  
    response_to_question_24: Literal["Yes", "No", "Not sure", "REFUSED", "OTHER"]  
    response_to_question_25: Literal["Yes", "No", "Not sure", "REFUSED", "OTHER"]  
    response_to_question_26: Literal["Yes", "No", "Not sure", "REFUSED", "OTHER"]  
    response_to_question_27: Literal["Yes", "No", "Not sure", "REFUSED", "OTHER"]  
    response_to_question_28: Literal["Yes", "No", "Not sure", "REFUSED", "OTHER"]  
    response_to_question_29: Literal["Yes", "No", "Not sure", "REFUSED", "OTHER"]  
    response_to_question_30: Literal["Yes", "No", "Not sure", "REFUSED", "OTHER"]  
    response_to_question_31: Literal["Yes", "No", "Not sure", "REFUSED", "OTHER"]  
  
    # Responses for questions 32-33  
    response_to_question_32: Literal["Yes", "No", "DON'T KNOW", "REFUSED", "OTHER"]  
    response_to_question_33: Literal["REFUSED", "OTHER"] | float 

In [5]:
client = AzureOpenAI(
  api_key = os.getenv('OPENAI_API_KEY'),  
  api_version = os.getenv('OPENAI_API_VERSION'),
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
)



role_description = f''' You are a diligent assistant tasked with analyzing survey conversation transcripts to deduce the respondent's answers to each of 
                        the {question_count} questions. Ensure that every question has a corresponding answer. Keep in mind the following guidelines:
                            
                            1. Deducing Answers: Responses to specific questions may sometimes be inferred from the broader conversation, even if the question was not explicitly asked. Use context to determine the best possible answer.
                            2. Handling Transcription Errors: Conversations may contain transcription errors. Do your best to interpret the respondent's intended meaning accurately.
                            3. Refusals or Skipped Questions: If the respondent explicitly refuses to answer or skips a question, label the answer as REFUSED. 
                               Keep in mind that if the respondent explicitly refuses to answer one question, this may sometimes imply a refusal to answer other related questions as well.
                            4. Ambiguity or Non-Matching Responses: If the respondent's answer does not align with any of the predefined response options or is ambiguous, label the answer as OTHER.

                        Your goal is to carefully analyze the transcript and assign the most appropriate answer to each question based on the above criteria.
                        
                        Below is the list of questions, each followed on the next line by its corresponding response options separated by semicolons:
                        {QA_details}
                        '''
with open(transcripts_file, 'r') as file:
    transcripts = json.load(file)

attempts_per_transcript = 3

def generate_response(conversation, question_count, attempts_per_transcript):
    question_to_reponses = {i:[] for i in range(question_count)} 
    for _ in range(attempts_per_transcript):

        response = client.beta.chat.completions.parse(
                model=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'), # model = "deployment_name".
                messages=conversation,
                response_format=SurveyResponses
            )
        responses_in_one_line = response.choices[0].message.content    

        reponse_dict = json.loads(responses_in_one_line)
        user_responses = []
        for question, answer in reponse_dict.items():
            user_responses.append(answer)
            
        '''    
        while len(user_responses) != question_count:
            print(f"Number of responses does not match expected number for {user_id}, got: {len(user_responses)}, expected: {question_count}")
            response = client.beta.chat.completions.parse(
                model=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'), # model = "deployment_name".
                messages=conversation,
                response_format=SurveyResponses
            )
            responses_in_one_line = response.choices[0].message.content    
    
            reponse_dict = json.loads(responses_in_one_line)
            user_responses = []
            for question, answer in reponse_dict.items():
                user_responses.append(answer)
        '''
        for i in range(question_count):
            answer = user_responses[i]
            question_to_reponses[i].append(answer)

    print("Here are all of the answers:")
    print(question_to_reponses)
    finalized_responses = []
    for i in range(question_count):
        counter = Counter(question_to_reponses[i])
        most_frequent = counter.most_common(1)[0][0]
        finalized_responses.append(most_frequent)

    print("Final response selections: ")
    print(finalized_responses)
    return finalized_responses

task_prompt = f"""Help me to understand the following conversation transcript : 

{transcripts["0"]}"""
#print(task_prompt)

conversation=[{"role": "system", "content": role_description}]
userid_to_answers = {}

for user_id, transcript in transcripts.items():
    survey = transcript.replace('user:', 'respondent:')
    survey = survey.replace('assistant:', 'surveyor:')
    task_prompt = f"""Help me to understand the following conversation transcript : 
                    {survey}"""
    conversation.append({"role": "user", "content": task_prompt})
    '''
    #print(task_prompt)
    #print(survey)
    
    response = client.beta.chat.completions.parse(
        model=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'), # model = "deployment_name".
        messages=conversation,
        response_format=SurveyResponses
    )
    responses_in_one_line = response.choices[0].message.content    
    
    reponse_dict = json.loads(responses_in_one_line)
    response_in_list = []
    for question, answer in reponse_dict.items():
        response_in_list.append(answer)
    #print(userid_to_answers[user_id])
    #print(response.choices[0].message.parsed)
    #print(response_in_list)
    #userid_to_answers[user_id] = response_in_list
    #person_responses = generate_response(conversation, question_count, attempts_per_transcript)
    #userid_to_answers[user_id] = person_responses
    '''
    person_responses = generate_response(conversation, question_count, attempts_per_transcript)
    userid_to_answers[user_id] = person_responses
    conversation.pop()

Here are all of the answers:
{0: ['45-54', '45-54', '45-54'], 1: ['Female', 'Female', 'Female'], 2: ['Guamanian or Chamorro', 'Guamanian or Chamorro', 'Guamanian or Chamorro'], 3: ['$100,000 to under $150,000', 'OTHER', '$100,000 to under $150,000'], 4: ['High school graduate - high school diploma or the equivalent', 'High school graduate - high school diploma or the equivalent', 'High school graduate - high school diploma or the equivalent'], 5: ['Four persons', 'Four persons', 'Four persons'], 6: [0, 0, 0], 7: [0, 0, 0], 8: [0, 0, 0], 9: [1, 1, 1], 10: [3, 3, 3], 11: ['A few times a week', 'A few times a week', 'A few times a week'], 12: ['Once a month', 'Once a month', 'Once a month'], 13: ['Not at all or less than 1 day', 'Not at all or less than 1 day', 'Not at all or less than 1 day'], 14: ['Not at all or less than 1 day', 'Not at all or less than 1 day', 'Not at all or less than 1 day'], 15: ['Not at all or less than 1 day', 'Not at all or less than 1 day', 'Not at all or less t

client = AzureOpenAI(
  api_key = os.getenv('OPENAI_API_KEY'),  
  api_version = os.getenv('OPENAI_API_VERSION'),
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
)



role_description = f''' You are a helpful assistant that reads survey conversation transcript and deduces responses given by user to each question.
                        There could have been errors while transcribing conversations.
                        You return an output with the responses for each question in order of they appear in the question list.
                        The returned output needs to be a SINGLE LINE, with individual responses separated by semicolons and no other punctuation.
                        Format of the output line should be: Question 1: Response 1; Question 2: Response 2; etc. 
                        Each deduced response for a question should be STRICTLY selected from corresponding Reponse Options. 
                        If Reponse Options includes 'Numeric Value' as an option, deduce actual numeric value from conversation.
                        Each question should have an answer, if question has Numeric Value as an option you couldn't deduce the response put 'NaN'.
                        If you are absolutely sure that there is no matching response option for a question and participant didn't explicitly refuse to answer then put 'NA'.
                        Make sure that number of answers equals number of questions.
                        
                        Below is the list of questions, each followed on the next line by its corresponding response options separated by semicolons:
                        {QA_details}
                        '''
#print(role_description)



with open(transcripts_file, 'r') as file:
    transcripts = json.load(file)



def generate_response(conversation, question_count, attempts_per_transcript):
    question_to_reponses = {i:[] for i in range(question_count)} 
    for _ in range(attempts_per_transcript):
        response = client.chat.completions.create(
            model=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'), # model = "deployment_name".
            messages=conversation
        )
        responses_in_one_line = response.choices[0].message.content
        user_responses = [part.strip() for part in responses_in_one_line.split(';')]
        while len(user_responses) != question_count:
            print(f"Number of responses does not match expected number for {user_id}, got: {len(user_responses)}, expected: {question_count}")
            response = client.chat.completions.create(
                model=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'), # model = "deployment_name".
                messages=conversation
            )
            user_responses = [part.strip() for part in userid_to_answers[user_id].split(';')]
        for i in range(question_count):
            response = user_responses[i]
            answer   = response[response.index(':')+1:].strip()
            answer   = 'nan' if answer == 'NaN' else answer
            question_to_reponses[i].append(answer)

    print("Here are all of the answers:")
    print(question_to_reponses)
    finalized_responses = []
    for i in range(question_count):
        counter = Counter(question_to_reponses[i])
        most_frequent = counter.most_common(1)[0][0]
        finalized_responses.append(most_frequent)

    print("Final response selections: ")
    print(finalized_responses)
    return finalized_responses
        
        

task_prompt = f"""Help me to understand the following conversation transcript : 

{transcripts["0"]}"""
#print(task_prompt)

conversation=[{"role": "system", "content": role_description}]
userid_to_answers = {}

attempts_per_transcript = 5

for user_id, transcript in transcripts.items():
    survey = transcript.replace('user:', 'respondent:')
    survey = survey.replace('assistant:', 'surveyor:')
    task_prompt = f"""Help me to understand the following conversation transcript : 
                    {survey}"""
    conversation.append({"role": "user", "content": task_prompt})

    #print(task_prompt)
    #print(survey)
    '''
    response = client.chat.completions.create(
        model=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'), # model = "deployment_name".
        messages=conversation
    )
    responses_in_one_line = response.choices[0].message.content    
    userid_to_answers[user_id] = responses_in_one_line
    print(userid_to_answers[user_id])'''

    person_responses = generate_response(conversation, question_count, attempts_per_transcript)
    userid_to_answers[user_id] = person_responses
    conversation.pop()

In [6]:
answers_as_list = []
'''
for user_id in sorted(list(userid_to_answers.keys())):
    user_responses = [part.strip() for part in userid_to_answers[user_id].split(';')]
    clean_user_responses = []
    for response in user_responses:
        clean_text = response[response.index(':')+1:].strip()
        if clean_text == 'NaN':
            clean_user_responses.append('nan')
        else:
            clean_user_responses.append(clean_text)
    answers_as_list.append(clean_user_responses)'''

for user_id in sorted(list(userid_to_answers.keys())):
    
    answers_as_list.append(userid_to_answers[user_id])
    print(len(userid_to_answers[user_id]))
    print(userid_to_answers[user_id])

33
['45-54', 'Female', 'Guamanian or Chamorro', '$100,000 to under $150,000', 'High school graduate - high school diploma or the equivalent', 'Four persons', 0, 0, 0, 1, 3, 'A few times a week', 'Once a month', 'Not at all or less than 1 day', 'Not at all or less than 1 day', 'Not at all or less than 1 day', 'Not at all or less than 1 day', 'Not at all or less than 1 day', "Yes, I worked for someone else for wages, salary, piece rate, commission, tips, or payments 'in kind,' for example, food or lodging received as payment for work performed", 'Good', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 99]
33
['55-64', 'Male', 'White', '$30,000 to under $40,000', '7th or 8th grade', 'REFUSED', "DON'T KNOW", "DON'T KNOW", "DON'T KNOW", "DON'T KNOW", "DON'T KNOW", 'A few times a week', 'Basically every day', 'Not at all or less than 1 day', 'Not at all or less than 1 day', '5-7 days', 'Not at all or less than 1 day', 'Not at all or less than 1 day', 'No, I did not wo

In [7]:
question_code_to_text_responses = {}
i = 0
#print(codebook['SOC1']['answer_to_answer_id']['Some'])
for code, val in codebook.items():
    print(code)
    question_code_to_text_responses[code] = []
    for user_answers in answers_as_list:
        response_text = user_answers[i]
        question_code_to_text_responses[code].append(response_text)
    i += 1

df = pd.DataFrame(question_code_to_text_responses)
df.head()

AGE7
GENDER
RACETH
HHINCOME
EDUCATION
HHSIZE1
HH01S
HH25S
HH612S
HH1317S
HH18OVS
SOC2A
SOC2B
SOC5A
SOC5B
SOC5C
SOC5D
SOC5E
ECON1
PHYS8
PHYS4
PHYS5
PHYS1B
PHYS1C
PHYS1D
PHYS1E
PHYS1F
PHYS1G
PHYS1H
PHYS1I
PHYS1J
PHYS11
PHYS11_TEMP


,AGE7,GENDER,RACETH,HHINCOME,EDUCATION,HHSIZE1,HH01S,HH25S,HH612S,HH1317S,...,PHYS1C,PHYS1D,PHYS1E,PHYS1F,PHYS1G,PHYS1H,PHYS1I,PHYS1J,PHYS11,PHYS11_TEMP
0,45-54,Female,Guamanian or Chamorro,"$100,000 to under $150,000",High school graduate - high school diploma or ...,Four persons,0,0,0,1,...,No,No,No,No,No,No,No,No,Yes,99
1,55-64,Male,White,"$30,000 to under $40,000",7th or 8th grade,REFUSED,DON'T KNOW,DON'T KNOW,DON'T KNOW,DON'T KNOW,...,No,No,No,No,No,No,No,No,Yes,95.3
2,45-54,Male,Filipino,"Under $10,000","Some college, no degree",Four persons,0,0,0,2,...,No,No,No,No,No,No,No,No,No,REFUSED
3,75+,Female,OTHER,"Under $10,000",Bachelor's degree,REFUSED,REFUSED,REFUSED,REFUSED,REFUSED,...,No,Yes,Yes,No,No,No,No,No,Yes,96.6
4,35-44,Female,White,"$75,000 to under $100,000",9th grade,Six or more persons,0,1,2,1,...,Yes,No,No,Yes,Yes,No,No,No,Yes,96.1


In [8]:
gpt_res_dir = os.path.join(os.getcwd(),  'gpt-deductions')
if not os.path.exists(gpt_res_dir):
    os.makedirs(gpt_res_dir)

In [9]:
# Save conversation transcripts
fname  = os.path.join(gpt_res_dir,  f"participant{participant_id}_deduced_{datetime.today().strftime('%Y-%m-%d')}.csv")
df.to_csv(fname, index=False)
df = pd.read_csv(fname)
#df = df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0', 'Unnamed: 0.2'])
df.head()

,AGE7,GENDER,RACETH,HHINCOME,EDUCATION,HHSIZE1,HH01S,HH25S,HH612S,HH1317S,...,PHYS1C,PHYS1D,PHYS1E,PHYS1F,PHYS1G,PHYS1H,PHYS1I,PHYS1J,PHYS11,PHYS11_TEMP
0,45-54,Female,Guamanian or Chamorro,"$100,000 to under $150,000",High school graduate - high school diploma or ...,Four persons,0,0,0,1,...,No,No,No,No,No,No,No,No,Yes,99
1,55-64,Male,White,"$30,000 to under $40,000",7th or 8th grade,REFUSED,DON'T KNOW,DON'T KNOW,DON'T KNOW,DON'T KNOW,...,No,No,No,No,No,No,No,No,Yes,95.3
2,45-54,Male,Filipino,"Under $10,000","Some college, no degree",Four persons,0,0,0,2,...,No,No,No,No,No,No,No,No,No,REFUSED
3,75+,Female,OTHER,"Under $10,000",Bachelor's degree,REFUSED,REFUSED,REFUSED,REFUSED,REFUSED,...,No,Yes,Yes,No,No,No,No,No,Yes,96.6
4,35-44,Female,White,"$75,000 to under $100,000",9th grade,Six or more persons,0,1,2,1,...,Yes,No,No,Yes,Yes,No,No,No,Yes,96.1
